# Reproducibility smoke test on the free sample

This notebook runs `../validation/benchmark_script.py`, **unmodified and identical to the script used for the published benchmark**, against the free sample shipped in `../sample/`.

**Purpose:** show that the code runs end to end against the real delivered schema.

**Not the purpose:** reproduce the published metric. The sample holds 250 of the 5,000 accounts, stratified by error type so the rarest error type has a single account. Each cross-validation fold has 50 accounts.

The number to cite is the one measured on the full dataset: detection AUC 0.9927 +/- 0.0042 (5-fold, patient-level CV, n = 5,000 accounts), 0.8646 without the three audit flags; 11 of 12 preregistered checks passed, with the failed P8 check published.


In [1]:
import sys, subprocess, shutil, tempfile, pathlib
import pandas as pd
HERE = pathlib.Path.cwd()
SAMPLE = (HERE / ".." / "sample").resolve()
SCRIPT = (HERE / ".." / "validation" / "benchmark_script.py").resolve()
work = pathlib.Path(tempfile.mkdtemp(prefix="smoke_"))   # all outputs go here, nothing is written into the repository
print("sample dir :", SAMPLE.name)
print("script     :", SCRIPT.name, "(unmodified copy of the published benchmark script)")
def clean(text):
    return text.replace(str(work), "<tmp>").replace(str(SAMPLE), "<sample>")


sample dir : sample
script     : benchmark_script.py (unmodified copy of the published benchmark script)


In [2]:
shutil.copytree(SAMPLE, work / "sample")
shutil.copy(SCRIPT, work / "benchmark_script.py")
r = subprocess.run([sys.executable, "benchmark_script.py", "--data", "sample", "--out", "."],
                   cwd=work, capture_output=True, text=True)
print(clean(r.stdout))
print(clean(r.stderr[-1500:]))
r.check_returncode()
print(sorted(p.name for p in work.iterdir()))

D06 benchmark | data=sample out=.
versions: xgboost 3.4.1, scikit-learn 1.9.1, pandas 3.0.5, numpy 2.5.3
  fold 1: AUC 0.9496  (n_test=50, prev=0.240)
  fold 2: AUC 0.9892  (n_test=50, prev=0.180)
  fold 3: AUC 0.9725  (n_test=50, prev=0.200)
  fold 4: AUC 0.9886  (n_test=50, prev=0.120)
  fold 5: AUC 1.0000  (n_test=50, prev=0.200)
  mean AUC: 0.9800 +/- 0.0176  (acceptance bar 0.87)
  naive AUC (no audit flags): 0.7539
wrote benchmark_results.parquet


['benchmark_results.parquet', 'benchmark_script.py', 'sample']


## Reading the output

The script's own output prints the per-fold AUC and the naive AUC without audit flags. On 250 accounts the AUC is high but the folds are small and the rarest error types have one to a handful of accounts, so use the published full-cohort figure for any claim. In the executed run the mean AUC is 0.980 (folds 0.950 to 1.000) and the AUC without the audit flags is 0.754, against the published 0.9927 and 0.8646.

In [3]:
results = pd.read_parquet(work / "benchmark_results.parquet")
results

,fold,auc,n_train,n_test,prevalence_test,naive_auc_no_audit_flags
0,1,0.949561,200.0,50.0,0.24,0.753885
1,2,0.989160,200.0,50.0,0.18,0.753885
2,3,0.972500,200.0,50.0,0.20,0.753885
3,4,0.988636,200.0,50.0,0.12,0.753885
4,5,1.000000,200.0,50.0,0.20,0.753885
5,mean,0.979972,NaN,NaN,NaN,0.753885
